In [1]:
#1
from pgmpy.models import DiscreteBayesianNetwork
from pgmpy.factors.discrete import TabularCPD
from pgmpy.inference import VariableElimination

model = DiscreteBayesianNetwork([("Suit", "Color"), ("Value", "IsFaceCard")])
suit_names = ["Hearts", "Diamonds", "Clubs", "Spades"]
cpd_suit = TabularCPD(
    variable="Suit", 
    variable_card=4, 
    values=[[0.25], [0.25], [0.25], [0.25]], 
    state_names={"Suit": suit_names}
)

cpd_value = TabularCPD(
    variable="Value", 
    variable_card=13, 
    values=[[1 / 13]] * 13,
    state_names={"Value": list(range(1,14))}
)

color_names = ["Red", "Black"]
cpd_color = TabularCPD(
    variable="Color",
    variable_card=2,
    values=[
        [1,1,0,0], 
        [0,0,1,1],
    ],
    evidence=["Suit"],
    evidence_card=[4],
    state_names={"Suit": suit_names, "Color": color_names},
)

face_probs = [0] * 10 + [1] * 3
not_face_probs = [1] * 10 + [0] * 3
cpd_face = TabularCPD(
    variable="IsFaceCard",
    variable_card=2,
    values=[
        not_face_probs,
        face_probs,
    ],
    evidence=["Value"],
    evidence_card=[13],
    state_names={"Value": list(range(1, 14)), "IsFaceCard": ["No", "Yes"]},
)

model.add_cpds(cpd_suit, cpd_value, cpd_color, cpd_face)

assert model.check_model()

infer = VariableElimination(model)

print("1. P(Red) = ?")
res1 = infer.query(variables=['Color'])
print(f"  > {res1.values[0]}") 

print("2. P(Heart | Red)= ?")
res2 = infer.query(variables=['Suit'], evidence={'Color': 'Red'})
print(f"  > {res2.values[0]}")

print("3. P(FaceCard | Diamond) = ?")
res3 = infer.query(variables=['IsFaceCard'], evidence={'Suit': 'Diamonds'})
print(f'  > {res3.values[1]}')

print("4. P(Spade or Queen | FaceCard) = ?")
res4 = infer.query(variables=['Suit', 'Value'], evidence={'IsFaceCard' : 'Yes'})
spade_prob = res4.values[3].sum()
queen_prob = res4.values[:, 11].sum()
print(f"  > {spade_prob + queen_prob}")

/home/f4ll3n/Work/Python/AI-Lab-Tasks/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/f4ll3n/Work/Python/AI-Lab-Tasks/.venv/lib/python3.14/site-packages/pgmpy/estimators/__init__.py:4: FutureWarning: `pgmpy.estimators.StructureScore` is deprecated and will be removed in v1.3.0. Use `pgmpy.structure_score` instead.
  from .StructureScore import (


1. P(Red) = ?
  > 0.5
2. P(Heart | Red)= ?
  > 0.5
3. P(FaceCard | Diamond) = ?
  > 0.23076923076923078
4. P(Spade or Queen | FaceCard) = ?
  > 0.5833333333333335


In [2]:
#2
from pgmpy.models import DiscreteBayesianNetwork
from pgmpy.factors.discrete import TabularCPD
from pgmpy.inference import VariableElimination

model = DiscreteBayesianNetwork([
    ('Intelligence', 'Grade'),
    ('StudyHours', 'Grade'),
    ('Difficulty', 'Grade'),
    ('Grade', 'Pass')
])

cpd_i = TabularCPD('Intelligence', 2, [[0.7], [0.3]], state_names={'Intelligence': ['High', 'Low']})

cpd_s = TabularCPD('StudyHours', 2, [[0.6], [0.4]], state_names={'StudyHours': ['Sufficient', 'Insufficient']})

cpd_d = TabularCPD('Difficulty', 2, [[0.4], [0.6]], state_names={'Difficulty': ['Hard', 'Easy']})

cpd_g = TabularCPD(
    'Grade', 3, 
    [
        # Intelligence: High | Low
        # StudyHours:   Suff | Insuff | Suff | Insuff
        # Difficulty:   H  E | H  E   | H  E | H  E
        [0.6, 0.9, 0.4, 0.7, 0.2, 0.5, 0.1, 0.3], # Grade A
        [0.3, 0.08, 0.4, 0.2, 0.4, 0.3, 0.3, 0.4], # Grade B
        [0.1, 0.02, 0.2, 0.1, 0.4, 0.2, 0.6, 0.3]  # Grade C
    ],
    evidence=['Intelligence', 'StudyHours', 'Difficulty'],
    evidence_card=[2, 2, 2],
    state_names={
        'Grade': ['A', 'B', 'C'],
        'Intelligence': ['High', 'Low'],
        'StudyHours': ['Sufficient', 'Insufficient'],
        'Difficulty': ['Hard', 'Easy']
    }
)

cpd_p = TabularCPD(
    'Pass', 2,
    [
        [0.95, 0.80, 0.50], # Pass: Yes
        [0.05, 0.20, 0.50]  # Pass: No
    ],
    evidence=['Grade'],
    evidence_card=[3],
    state_names={'Pass': ['Yes', 'No'], 'Grade': ['A', 'B', 'C']}
)

model.add_cpds(cpd_i, cpd_s, cpd_d, cpd_g, cpd_p)
assert model.check_model()

infer = VariableElimination(model)

print("P(Passes | Sufficient study, Hard difficulty) :")
result_a = infer.query(variables=['Pass'], evidence={'StudyHours': 'Sufficient', 'Difficulty': 'Hard'})
print(result_a)

print("\nP(High Intelligence | student passed) :")
result_b = infer.query(variables=['Intelligence'], evidence={'Pass': 'Yes'})
print(result_b)

P(Passes | Sufficient study, Hard difficulty) :
+-----------+-------------+
| Pass      |   phi(Pass) |
+===========+=============+
| Pass(Yes) |      0.8150 |
+-----------+-------------+
| Pass(No)  |      0.1850 |
+-----------+-------------+

P(High Intelligence | student passed) :
+--------------------+---------------------+
| Intelligence       |   phi(Intelligence) |
+====================+=====================+
| Intelligence(High) |              0.7331 |
+--------------------+---------------------+
| Intelligence(Low)  |              0.2669 |
+--------------------+---------------------+


In [3]:
#3
from pgmpy.models import DiscreteBayesianNetwork
from pgmpy.factors.discrete import TabularCPD
from pgmpy.inference import VariableElimination

model = DiscreteBayesianNetwork([
    ('Disease', 'Fever'),
    ('Disease', 'Cough'),
    ('Disease', 'Fatigue'),
    ('Disease', 'Chills')
])

cpd_disease = TabularCPD(
    variable='Disease', variable_card=2, 
    values=[[0.3], [0.7]],
    state_names={'Disease': ['Flu', 'Cold']}
)

cpd_fever = TabularCPD(
    variable='Fever', variable_card=2,
    values=[[0.9, 0.5],  # Yes
            [0.1, 0.5]], # No
    evidence=['Disease'], evidence_card=[2],
    state_names={'Fever': ['Yes', 'No'], 'Disease': ['Flu', 'Cold']}
)

cpd_cough = TabularCPD(
    variable='Cough', variable_card=2,
    values=[[0.8, 0.6],  # Yes
            [0.2, 0.4]], # No
    evidence=['Disease'], evidence_card=[2],
    state_names={'Cough': ['Yes', 'No'], 'Disease': ['Flu', 'Cold']}
)

cpd_fatigue = TabularCPD(
    variable='Fatigue', variable_card=2,
    values=[[0.7, 0.3],  # Yes
            [0.3, 0.7]], # No
    evidence=['Disease'], evidence_card=[2],
    state_names={'Fatigue': ['Yes', 'No'], 'Disease': ['Flu', 'Cold']}
)

cpd_chills = TabularCPD(
    variable='Chills', variable_card=2,
    values=[[0.6, 0.4],  # Yes
            [0.4, 0.6]], # No
    evidence=['Disease'], evidence_card=[2],
    state_names={'Chills': ['Yes', 'No'], 'Disease': ['Flu', 'Cold']}
)

model.add_cpds(cpd_disease, cpd_fever, cpd_cough, cpd_fatigue, cpd_chills)
assert model.check_model()
infer = VariableElimination(model)

print("P(Disease | Fever=Yes, Cough=Yes) : ")
print(infer.query(variables=['Disease'], evidence={'Fever': 'Yes', 'Cough': 'Yes'}))

print("\nP(Disease | Fever=Yes, Cough=Yes, Chills=Yes) : ")
print(infer.query(variables=['Disease'], evidence={'Fever': 'Yes', 'Cough': 'Yes', 'Chills': 'Yes'}))

print("\nP(Fatigue=Yes | Disease=Flu) : ")
print(infer.query(variables=['Fatigue'], evidence={'Disease': 'Flu'}))

P(Disease | Fever=Yes, Cough=Yes) : 
+---------------+----------------+
| Disease       |   phi(Disease) |
+===============+================+
| Disease(Flu)  |         0.5070 |
+---------------+----------------+
| Disease(Cold) |         0.4930 |
+---------------+----------------+

P(Disease | Fever=Yes, Cough=Yes, Chills=Yes) : 
+---------------+----------------+
| Disease       |   phi(Disease) |
+===============+================+
| Disease(Flu)  |         0.6067 |
+---------------+----------------+
| Disease(Cold) |         0.3933 |
+---------------+----------------+

P(Fatigue=Yes | Disease=Flu) : 
+--------------+----------------+
| Fatigue      |   phi(Fatigue) |
+==============+================+
| Fatigue(Yes) |         0.7000 |
+--------------+----------------+
| Fatigue(No)  |         0.3000 |
+--------------+----------------+


In [4]:
#4
import numpy as np

states = ["Sunny", "Cloudy", "Rainy"]
state_map = {0: "Sunny", 1: "Cloudy", 2: "Rainy"}

transition_matrix = np.array([
    [0.7, 0.2, 0.1], # Sunny
    [0.3, 0.4, 0.3], # Cloudy
    [0.2, 0.3, 0.5]  # Rainy
])

def simulate_weather(start_state, days):
    current_state = start_state
    weather_sequence = [state_map[current_state]]
    
    for _ in range(days - 1):
        current_state = np.random.choice([0, 1, 2], p=transition_matrix[current_state])
        weather_sequence.append(state_map[current_state])
        
    return weather_sequence

np.random.seed(32)
ten_day_forecast = simulate_weather(0, 10) # 0 = Sunny
print(f"10-Day Simulation: {ten_day_forecast}")

trials = 10000
at_least_3_rainy = 0

for _ in range(trials):
    sim = simulate_weather(0, 10)
    if sim.count("Rainy") >= 3:
        at_least_3_rainy += 1

probability = at_least_3_rainy / trials
print(f"\nProbability of at least 3 rainy days in 10 days: {probability:.2%}")

10-Day Simulation: ['Sunny', 'Cloudy', 'Cloudy', 'Cloudy', 'Rainy', 'Rainy', 'Rainy', 'Sunny', 'Rainy', 'Rainy']

Probability of at least 3 rainy days in 10 days: 35.24%
